# Week 2: Connect to Your Data (RAG Intro)

**Lab companion to [week_02.md](../agenda/week_02.md)**

In this lab, you will:
1. Load documents from files
2. Search through your documents
3. Build a simple Q&A system
4. Understand why we need embeddings (next week)

In [14]:
# Setup
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI()

print("✓ Ready!")

✓ Ready!


## 1. The Problem: AI Doesn't Know Your Data

GPT was trained on public internet data. It doesn't know about:
- Your company's internal documents
- Recent events (after training cutoff)
- Private information

In [15]:
# Let's create some sample "company documents"
documents = [
    {
        "title": "Company Policy - Remote Work",
        "content": """Employees may work remotely up to 3 days per week.
Remote work requests must be approved by direct managers.
All remote employees must be available during core hours (10am-4pm).
VPN must be used when accessing company systems remotely."""
    },
    {
        "title": "Company Policy - PTO",
        "content": """Full-time employees receive 20 days of PTO per year.
PTO requests must be submitted at least 2 weeks in advance.
Unused PTO can be carried over up to 5 days to the next year.
PTO is accrued monthly at a rate of 1.67 days per month."""
    },
    {
        "title": "Company Policy - Expenses",
        "content": """Business expenses must be submitted within 30 days.
Receipts are required for all expenses over $25.
Travel expenses require pre-approval from department heads.
Meal expenses are limited to $50 per person per day."""
    },
    {
        "title": "IT Support Guide",
        "content": """For password resets, use the self-service portal at passwords.company.com.
For hardware issues, submit a ticket to it-support@company.com.
Emergency IT support is available 24/7 at extension 5555.
New software requests require manager approval."""
    }
]

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  - {doc['title']}")

Loaded 4 documents
  - Company Policy - Remote Work
  - Company Policy - PTO
  - Company Policy - Expenses
  - IT Support Guide


## 2. Naive Approach: Stuff All Context

The simplest RAG: put everything in the prompt

In [16]:
def ask_with_all_context(question: str, docs: list) -> str:
    """Answer a question using ALL documents as context."""

    # Build context from all documents
    context = "\n\n".join([
        f"=== {doc['title']} ===\n{doc['content']}"
        for doc in docs
    ])

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {
                "role": "system",
                "content": "Answer questions based only on the provided documents. If the answer isn't in the documents, say so."
            },
            {
                "role": "user",
                "content": f"""Documents:
{context}

Question: {question}"""
            }
        ]
    )

    return response.choices[0].message.content

# Test
print(ask_with_all_context("How many PTO days do I get?", documents))

The policy states full-time employees receive 20 days of PTO per year. PTO is accrued monthly at 1.67 days/month, unused PTO can be carried over up to 5 days, and PTO requests must be submitted at least 2 weeks in advance. 

If you are not a full-time employee, the provided documents do not say how many PTO days apply.


In [17]:
# More questions
questions = [
    "How do I reset my password?",
    "Can I work from home?",
    "What's the expense limit for meals?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask_with_all_context(q, documents)}")
    print()

Q: How do I reset my password?
A: Use the self-service portal at passwords.company.com to reset your password. If the portal doesn’t work or you need further help, submit a ticket to it-support@company.com or call emergency IT support at extension 5555.

Q: Can I work from home?
A: Yes. Employees may work remotely up to 3 days per week, but remote work requests must be approved by your direct manager. Remote employees must be available during core hours (10am–4pm) and must use the VPN when accessing company systems.

Q: What's the expense limit for meals?
A: Meal expenses are limited to $50 per person per day.



## 3. The Problem with "Stuff Everything"

This works for small document sets, but has issues:

In [18]:
import tiktoken

def count_tokens(text: str) -> int:
    encoding = tiktoken.encoding_for_model("gpt-5-mini")
    return len(encoding.encode(text))

# Current document size
all_text = "\n".join([d['content'] for d in documents])
current_tokens = count_tokens(all_text)
print(f"Current documents: {current_tokens} tokens")

# What if we had 100 documents?
estimated_100_docs = current_tokens * 25
print(f"100 documents: ~{estimated_100_docs:,} tokens")

# What if we had 1000 documents?
estimated_1000_docs = current_tokens * 250
print(f"1000 documents: ~{estimated_1000_docs:,} tokens")

print(f"\ngpt-5-mini context limit: 128,000 tokens")
print(f"100 docs would use {estimated_100_docs/128000*100:.1f}% of context")

Current documents: 200 tokens
100 documents: ~5,000 tokens
1000 documents: ~50,000 tokens

gpt-5-mini context limit: 128,000 tokens
100 docs would use 3.9% of context


## 4. Better Approach: Search First, Then Ask

Only include RELEVANT documents in the context.

In [19]:
def keyword_search(query: str, docs: list, top_k: int = 2) -> list:
    """Simple keyword-based search."""

    # Score each document by keyword matches
    query_words = set(query.lower().split())

    scored = []
    for doc in docs:
        doc_text = (doc['title'] + ' ' + doc['content']).lower()

        # Count matching words
        score = sum(1 for word in query_words if word in doc_text)
        scored.append((doc, score))

    # Sort by score and return top_k
    scored.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, score in scored[:top_k] if score > 0]

# Test
results = keyword_search("password reset", documents)
print("Search results for 'password reset':")
for doc in results:
    print(f"  - {doc['title']}")

Search results for 'password reset':
  - IT Support Guide


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Keyword search is simple:
1. Split query into words
2. Count how many words appear in each document
3. Sort by count, return top results

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
def keyword_search(query: str, docs: list, top_k: int = 2) -> list:
    query_words = set(query.lower().split())

    scored = []
    for doc in docs:
        doc_text = (doc['title'] + ' ' + doc['content']).lower()
        score = sum(1 for word in query_words if word in doc_text)
        scored.append((doc, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, score in scored[:top_k] if score > 0]
```

</details>

In [21]:
def ask_with_search(question: str, docs: list) -> str:
    """Answer using only relevant documents."""

    # Search for relevant documents
    relevant = keyword_search(question, docs, top_k=2)

    if not relevant:
        return "I couldn't find any relevant information in the documents."

    # Build context from relevant docs only
    context = "\n\n".join([
        f"=== {doc['title']} ===\n{doc['content']}"
        for doc in relevant
    ])

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {
                "role": "system",
                "content": "Answer questions based only on the provided documents."
            },
            {
                "role": "user",
                "content": f"""Documents:
{context}

Question: {question}"""
            }
        ]
    )

    return response.choices[0].message.content

# Test
print(ask_with_search("How do I reset my password?", documents))

Use the self-service portal at passwords.company.com to reset your password. If you run into problems, contact IT (it-support@company.com) or, for emergencies, call extension 5555.


## 5. The Limitation of Keyword Search

In [23]:
# Keyword search fails with synonyms
test_queries = [
    "How many vacation days do I get?",  # "vacation" vs "PTO"
    "Can I telecommute?",                 # "telecommute" vs "remote work"
    "What are the rules for reimbursement?"  # "reimbursement" vs "expenses"
]

for query in test_queries:
    results = keyword_search(query, documents)
    print(f"Query: '{query}'")
    if results:
        print(f"  Found: {[r['title'] for r in results]}")
    else:
        print("  No results! (keyword mismatch)")
    print()

Query: 'How many vacation days do I get?'
  Found: ['Company Policy - Remote Work', 'Company Policy - PTO']

Query: 'Can I telecommute?'
  Found: ['Company Policy - PTO', 'Company Policy - Remote Work']

Query: 'What are the rules for reimbursement?'
  Found: ['IT Support Guide', 'Company Policy - Expenses']



## 6. Preview: Semantic Search with Embeddings

This is what we'll build in Week 4:

In [24]:
import numpy as np

def get_embedding(text: str) -> list:
    """Get embedding vector for text."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

def cosine_similarity(a: list, b: list) -> float:
    """Calculate similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Create embeddings for all documents
print("Creating embeddings...")
for doc in documents:
    doc['embedding'] = get_embedding(doc['title'] + " " + doc['content'])
print("Done!")

Creating embeddings...
Done!


In [30]:
def semantic_search(query: str, docs: list, top_k: int = 2) -> list:
    """Search using semantic similarity."""

    query_embedding = get_embedding(query)

    # Score by similarity
    scored = []
    for doc in docs:
        score = cosine_similarity(query_embedding, doc['embedding'])
        scored.append((doc, score))

    # Sort by score
    scored.sort(key=lambda x: x[1], reverse=True)
    # print(f'scored: {scored}')
    return [(doc, score) for doc, score in scored[:top_k]]

# Test with the same queries that failed keyword search
for query in test_queries:
    results = semantic_search(query, documents)
    print(f"Query: '{query}'")
    for doc, score in results:
        print(f"  {doc['title']}: {score:.3f}")
    print()

Query: 'How many vacation days do I get?'
  Company Policy - PTO: 0.454
  Company Policy - Remote Work: 0.337

Query: 'Can I telecommute?'
  Company Policy - Remote Work: 0.530
  Company Policy - PTO: 0.266

Query: 'What are the rules for reimbursement?'
  Company Policy - Expenses: 0.491
  Company Policy - Remote Work: 0.283



## 7. Build a Simple RAG Pipeline

In [31]:
class SimpleRAG:
    """A simple RAG system."""

    def __init__(self):
        self.client = OpenAI()
        self.documents = []

    def add_document(self, title: str, content: str):
        """Add a document to the knowledge base."""
        embedding = self._embed(title + " " + content)
        self.documents.append({
            'title': title,
            'content': content,
            'embedding': embedding
        })

    def _embed(self, text: str) -> list:
        response = self.client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding

    def _similarity(self, a: list, b: list) -> float:
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    def search(self, query: str, top_k: int = 2) -> list:
        """Search for relevant documents."""
        query_emb = self._embed(query)

        scored = [
            (doc, self._similarity(query_emb, doc['embedding']))
            for doc in self.documents
        ]
        scored.sort(key=lambda x: x[1], reverse=True)

        return scored[:top_k]

    def ask(self, question: str) -> str:
        """Answer a question using RAG."""

        # Retrieve
        results = self.search(question, top_k=2)

        if not results:
            return "No relevant documents found."

        # Build context
        context = "\n\n".join([
            f"Source: {doc['title']}\n{doc['content']}"
            for doc, score in results
        ])

        # Generate
        response = self.client.chat.completions.create(
            model="gpt-5-mini",
            messages=[
                {
                    "role": "system",
                    "content": "Answer based on the provided documents. Cite your sources."
                },
                {
                    "role": "user",
                    "content": f"Documents:\n{context}\n\nQuestion: {question}"
                }
            ]
        )

        return response.choices[0].message.content

# Build RAG system
rag = SimpleRAG()
for doc in documents:
    rag.add_document(doc['title'], doc['content'])

print(f"RAG system ready with {len(rag.documents)} documents")

RAG system ready with 4 documents


In [32]:
# Test the RAG system
test_questions = [
    "How many vacation days do I get per year?",
    "Can I work from home? What are the requirements?",
    "How do I get my computer fixed?",
    "What's the per-day limit for food expenses on business trips?"
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {rag.ask(q)}")
    print("-" * 50)

Q: How many vacation days do I get per year?
A: Full‑time employees receive 20 days of PTO (vacation) per year. (Company Policy - PTO)

Additional relevant details: 
- PTO accrues monthly at 1.67 days/month. (Company Policy - PTO)  
- Unused PTO can be carried over up to 5 days to the next year. (Company Policy - PTO)  
- PTO requests must be submitted at least 2 weeks in advance. (Company Policy - PTO)
--------------------------------------------------
Q: Can I work from home? What are the requirements?
A: Yes — you can work from home, subject to the company’s remote-work rules.

Requirements
- Remote work is allowed up to 3 days per week. (Company Policy - Remote Work)
- Remote-work requests must be approved by your direct manager. (Company Policy - Remote Work)
- You must be available during core hours: 10:00 AM–4:00 PM. (Company Policy - Remote Work)
- Use the company VPN when accessing company systems remotely. (Company Policy - Remote Work)

If you incur business expenses while r

## 8. Loading Real Files

Load documents from actual files:

In [40]:
import os
from pathlib import Path

def load_text_files(folder: str) -> list:
    """Load all .txt files from a folder."""
    documents = []
    folder_path = Path(folder)

    if not folder_path.exists():
        print(f"Folder {folder} doesn't exist")
        return []

    for file_path in folder_path.glob("*.txt"):
        content = file_path.read_text(encoding='utf-8')
        documents.append({
            'title': file_path.stem,
            'content': content,
            'source': str(file_path)
        })

    return documents

def load_markdown_files(folder: str) -> list:
    """Load all .md files from a folder."""
    documents = []
    folder_path = Path(folder)

    for file_path in folder_path.glob("**/*_1*.md"):  # Recursive
        content = file_path.read_text(encoding='utf-8')

        # Extract title from first heading
        lines = content.split('\n')
        title = file_path.stem
        for line in lines:
            if line.startswith('# '):
                title = line[2:].strip()
                break

        documents.append({
            'title': title,
            'content': content,
            'source': str(file_path)
        })

    return documents

# Example usage (modify path to your actual files)
docs = load_markdown_files("../agenda")
print(f"Loaded {len(docs)} files")
short_info = "\n".join(f"Title: {d['title']}, source: {d['source']}" for d in docs)
print(f'Short_info: \n{short_info}')
# print(f"docs: {docs}")

Loaded 6 files
Short_info: 
Title: Week 14: Deployment & Fine-Tuning, source: ../agenda/week_14.md
Title: Week 10: LangGraph in Practice + LangSmith & Langfuse, source: ../agenda/week_10.md
Title: Week 11: Data Engineering for AI + Portfolio Project #2, source: ../agenda/week_11.md
Title: Week 15: Capstone Project & Interview Prep, source: ../agenda/week_15.md
Title: Week 12: MCP & Observability, source: ../agenda/week_12.md
Title: Week 13: Production Hardening, source: ../agenda/week_13.md


## Summary

You learned:
- ✅ Why AI needs your documents (it doesn't know your data)
- ✅ Simple approach: stuff all context
- ✅ Better approach: search then ask
- ✅ Keyword search limitations
- ✅ Preview of semantic search with embeddings
- ✅ Built a simple RAG pipeline

**Next:** [Week 3 - Embeddings Deep Dive](week_03_embeddings.ipynb)